# Cube Thermal Steady-State Demo

Example usage of `CubeThermalSteadyStateSimulator` on a small batch of samples.

In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from digilab_simulators.simulators import (
    CubeThermalSteadyStateConfig,
    simulator_factory,
)

config = CubeThermalSteadyStateConfig(
    nx=8,
    ny=8,
    nz=8,
    length_x=1.0,
    length_y=0.75,
    length_z=0.5,
    ambient_temperature=293.15,
    max_iterations=1500,
    tolerance=1e-5,
    initialisation="linear_hot_to_cold",
)

simulator = simulator_factory(config)
config


In [ ]:
import pandas as pd

X = pd.DataFrame(
    [
        {
            "thermal_conductivity": 12.0,
            "volumetric_heat_source": 2.0e4,
            "heat_flux_x0": 1.5e4,
            "convective_h": 8.0,
            "initial_temperature": 293.15,
        },
        {
            "thermal_conductivity": 18.0,
            "volumetric_heat_source": 3.0e4,
            "heat_flux_x0": 2.0e4,
            "convective_h": 12.0,
            "initial_temperature": 295.15,
            "length_x": 1.2,
        },
        {
            "thermal_conductivity": 25.0,
            "volumetric_heat_source": 1.0e4,
            "heat_flux_x0": 8.0e3,
            "convective_h": 20.0,
            "initial_temperature": 290.15,
            "length_y": 1.0,
            "length_z": 0.8,
        },
    ]
)

X


In [ ]:
outputs = simulator.forward(X)
len(outputs), outputs[0].keys()


In [ ]:
summary_df = simulator.outputs_to_summary_dataframe(outputs)
summary_df.round(3)


In [ ]:
sample_index = 0
point_df = simulator.output_to_point_dataframe(outputs[sample_index])
point_df.head()


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].bar(summary_df["sample_index"], summary_df["centre_temperature"])
ax[0].set_xlabel("Sample")
ax[0].set_ylabel("Centre temperature (K)")
ax[0].set_title("Centre temperature by sample")

ax[1].bar(summary_df["sample_index"], summary_df["temperature_range"], color="tomato")
ax[1].set_xlabel("Sample")
ax[1].set_ylabel("Temperature range (K)")
ax[1].set_title("Temperature range by sample")

plt.tight_layout()


In [ ]:
import numpy as np

sample_output = outputs[sample_index]
nx = sample_output["nx"]
ny = sample_output["ny"]
nz = sample_output["nz"]

temperature = np.array(sample_output["point_data"]["temperature"]).reshape(nx, ny, nz)
mid_k = nz // 2
slice_xy = temperature[:, :, mid_k]

plt.figure(figsize=(6, 5))
plt.imshow(slice_xy.T, origin="lower", aspect="auto", cmap="inferno")
plt.colorbar(label="Temperature (K)")
plt.xlabel("x node index")
plt.ylabel("y node index")
plt.title(f"Mid-plane temperature slice for sample {sample_index}")
plt.tight_layout()
